# Pre-processing

## Strut'n'Tie — LUSAS LPI
### Video 05

**Author:** Kamil Riedel  
**© Strut'n'Tie**  
**License:** MIT Licence

# Connect to LUSAS

In [1]:
from shared.LPI import *
import shared.Helpers as Helpers

modeller = get_lusas_modeller()
Helpers.initialise(modeller)

if not modeller.existsDatabase():
    raise Exception("A model must be open before running this code")

database = modeller.database()

modeller.getTextWindow().writeLine("Hello World")

# Define variables

In [2]:
# ======================================================= #
# STRING CONSTANTS #
BEAM_MESH = "truss mesh"
STEEL = "steel"
BOTTOM_CHORD = "bottom chord"
TOP_CHORD = "top chord"
DIAGONALS = "diagonals"
PINNED = "pinned"
ROLLER = "roller"
LOAD_DL = "Point load DL"
LOAD_LL = "Point load LL"
ANALYSIS_MAIN = "ULS"
LOADCASE_SW = "Self-weight"
LOADCASE_DL = "Dead load"
LOADCASE_LL = "Live load"

In [3]:
# ======================================================= #
# DICTIONARIES TO DEFINE TRUSS #
points_coord = { 1: (0.0, 0), 2: (1.5, 3), 3: (3.0, 0), 4: (4.5, 3), 5: (6.0, 0), 6: (7.5, 3), 7: (9.0, 0), 8: (10.5, 3), 9: (12.0, 0),     10: (13.5, 3), 11: (15.0, 0),    12: (16.5, 3), 13: (18.0, 0),    14: (19.5, 3), 15: (21.0, 0),    16: (22.5, 3), 17: (24.0, 0),    18: (25.5, 3), 19: (27.0, 0) }
elm_connectivity = { 1:(1,3), 2:(3,5), 3:(5,7), 4:(7,9), 5:(9,11), 6:(11,13), 7:(13,15), 8:(15,17), 9:(17,19), 10:(2,4), 11:(4,6), 12:(6,8), 13:(8,10), 14:(10,12), 15:(12,14), 16:(14,16), 17:(16,18), 18:(1,2), 19:(2,3), 20:(3,4), 21:(4,5), 22:(5,6), 23:(6,7), 24:(7,8), 25:(8,9), 26:(9,10), 27:(10,11), 28:(11,12), 29:(12,13), 30:(13,14), 31:(14,15), 32:(15,16), 33:(16,17), 34:(17,18), 35:(18,19) }
sections = { 1: BOTTOM_CHORD, 2: BOTTOM_CHORD, 3: BOTTOM_CHORD, 4: BOTTOM_CHORD, 5: BOTTOM_CHORD, 6: BOTTOM_CHORD, 7: BOTTOM_CHORD, 8: BOTTOM_CHORD, 9: BOTTOM_CHORD, 10: TOP_CHORD, 11: TOP_CHORD, 12: TOP_CHORD, 13: TOP_CHORD, 14: TOP_CHORD, 15: TOP_CHORD, 16: TOP_CHORD, 17: TOP_CHORD, 18: DIAGONALS, 19: DIAGONALS, 20: DIAGONALS, 21: DIAGONALS, 22: DIAGONALS, 23: DIAGONALS, 24: DIAGONALS, 25: DIAGONALS, 26: DIAGONALS, 27: DIAGONALS, 28: DIAGONALS, 29: DIAGONALS, 30: DIAGONALS, 31: DIAGONALS, 32: DIAGONALS, 33: DIAGONALS, 34: DIAGONALS, 35: DIAGONALS }
supports = {1 : PINNED, 19 : ROLLER}
loads = [3,5,7,9,11,13,15,17]

# Create analyses

In [4]:
# ======================================================= #
# ANALYSES #

# Rename the first analysis
analysis = database.getAnalyses()[0]
analysis.setName(ANALYSIS_MAIN)
# print(analysis.getName())

# Rename the first loadcase
loadcase_SW = database.getLoadsets("All", "All")[0]
loadcase_SW.setName(LOADCASE_SW)

# Enable gravity in self-weight loadcase
loadcase_SW.addGravity(True)
loadcase_SW.setGravityFactor(1.0)
database.setVerticalDir("Y")

# Create additional loadcases
loadcase_DL = database.createLoadcase(LOADCASE_DL, ANALYSIS_MAIN)
loadcase_LL = database.createLoadcase(LOADCASE_LL, ANALYSIS_MAIN)

# Create geometry

In [5]:
# ======================================================= #
# DEFINE GEOMETRY #

# Create points
point_objects = {}
for pointID, (x, y) in points_coord.items():
    # geometry_data = modeller.geometryData().setAllDefaults()
    # geometry_data.addCoords(x, y, 0.0)
    # geometry_data.setLowerOrderGeometryType("coordinates")
    # point = database.createPoint(geometry_data)
    point = Helpers.create_point(x, y, 0)
    point_objects[pointID] = point

# Create lines
line_objects = {}
for elmID, (p1ID, p2ID) in elm_connectivity.items():
    # point1, point2 = database.getObject("Point", p1ID), database.getObject("Point", p2ID)
    point1, point2 = point_objects[p1ID], point_objects[p2ID]
    line = Helpers.create_line_from_points(point1, point2)
    line_objects[elmID] = line

# Create attributes

In [6]:
# ======================================================= #
# CREATE ATTRIBUTES #

# Mesh attributes
meshAttr = database.createMeshLine(BEAM_MESH)
meshAttr.setSize("BMI21", 1)
meshAttr.setEndRelease("Start", "THZ", "free")
meshAttr.setEndRelease("End", "THZ", "free")

# Material attributes
matAttr = database.createIsotropicMaterial(STEEL, 210.0E9, 0.3, 7.84913E3).setValue("alpha", 12.0E-6, 0)
matAttr.setDefinitionMenuID(1, None, True)
matAttr.setDescription("Ungraded | Steel - Structural | EN1993-1-1:2005")
matAttr.createValue("Region", 0, 0, 0, 0, 0, 0, 0).setValue("Region", "UK")
matAttr.createValue("Standard", 0, 0, 0, 0, 0, 0, 0).setValue("Standard", "EN1993-1-1:2005")
matAttr.createValue("Material", 0, 0, 0, 0, 0, 0, 0).setValue("Material", "Steel - Structural")
matAttr.createValue("Grade", 0, 0, 0, 0, 0, 0, 0).setValue("Grade", "Ungraded")

# Geometric attributes
geomAttrTopChords = database.createGeometricLine(TOP_CHORD)
geomAttrTopChords.setValue("elementType", "3D Thick Beam")
geomAttrTopChords.setFromLibrary("UK Sections", "Universal Columns (Advance)", "254x254x73kg UKUC", 0, 0, 0)

geomAttrBottomChords = database.createGeometricLine(BOTTOM_CHORD)
geomAttrBottomChords.setValue("elementType", "3D Thick Beam")
geomAttrBottomChords.setFromLibrary("UK Sections", "Universal Columns (Advance)", "254x254x73kg UKUC", 0, 0, 0)

geomAttrDiagonals = database.createGeometricLine(DIAGONALS)
geomAttrDiagonals.setValue("elementType", "3D Thick Beam")
geomAttrDiagonals.setFromLibrary("UK Sections", "Universal Columns (Advance)", "152x152x30kg UKUC", 0, 0, 0)

# Support attributes
supportPinned = database.createSupportStructural(PINNED)
supportPinned.setStructural("R", "R", "R", "R", "R", "F", "F", "F", "C", "F")

supportRoller = database.createSupportStructural(ROLLER)
supportRoller.setStructural("R", "R", "R", "R", "R", "F", "F", "F", "C", "F")

# Load attributes
loadAttrDL = database.createLoadingConcentrated(LOAD_DL)
loadAttrDL.setConcentrated(0.0, "-80.0E3", 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)

loadAttrLL = database.createLoadingConcentrated(LOAD_LL)
loadAttrLL.setConcentrated(0.0, "-50.0E3", 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)

<COMObject setConcentrated>

# Assign attributes

In [7]:
# ======================================================= #
# ASSIGN ATTRIBUTES #

for elmID, line in line_objects.items():
    # Assign mesh attribute
    # meshAttr.assignTo(line)
    assignment = modeller.newAssignment()
    assignment.setAllDefaults()
    assignment.setLoadset(LOADCASE_SW)
    assignment.setBetaAngle("90.0")
    assignment.meshSecondaryToPrimary()
    assignment.setSingleFeatureJointOrient("axes")
    meshAttr.assignTo(line, assignment)

    # Assign geometric attribute
    # if sections[elmID] == TOP_CHORD:
    #     geomAttrTopChords.assignTo(line)
    # elif sections[elmID] == BOTTOM_CHORD:
    #     geomAttrBottomChords.assignTo(line)
    # elif sections[elmID] == DIAGONALS:
    #     geomAttrDiagonals.assignTo(line)
    # else:
    #     raise Exception("Error - section name")
    geomAttr = database.getAttribute("Geometric", sections[elmID])
    geomAttr.assignTo(line)

    # Assign material attribute
    matAttr.assignTo(line)
    # matAttr.assignTo("line", elmID)
    # matAttr.assignTo("line", line.getName())
database.updateMesh()

# Assign supports
for pointID, supportName in supports.items():
    supportAttr = database.getAttribute("Support", supportName)
    supportAttr.assignTo(point_objects[pointID])

# Assign point loads
assignDL = modeller.newAssignment().setAllDefaults()
assignDL.setLoadset(loadcase_DL)  
assignLL = modeller.newAssignment().setAllDefaults()
assignLL.setLoadset(loadcase_LL) 
for pointID in loads:
    loadAttrDL.assignTo(point_objects[pointID], assignDL)
    loadAttrLL.assignTo(point_objects[pointID], assignLL)

# Solve

In [8]:
# ======================================================= #
# SOLVE #

# Run the analysis
database.getAnalysis(ANALYSIS_MAIN).solve(False)

# Open available results
database.openAllResults(False)